# Part 7 — Time-Series Foundation Model (Chronos-Bolt, zero-shot)
**Appliance Energy Use Forecasting — 7PAM2033**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DolapoMichael/Time-Series-coding-Case-study-and-Report/blob/main/notebooks/07_foundation_model.ipynb)

Uses Amazon's Chronos-Bolt (`amazon/chronos-bolt-small`) as a **zero-shot, target-only** forecaster: no fine-tuning on this dataset, no sensor/weather covariates — purely the model's pretrained knowledge from a large, diverse corpus of other time series, given only the training portion of this series as context. This is the simplest of the brief's four allowed usage modes.

Requires internet access to `huggingface.co` to download pretrained weights on first run — this should work without changes on Colab's default runtime.

In [ ]:
!pip install -q chronos-forecasting

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from chronos import BaseChronosPipeline

RAW_CSV_URL = 'https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv'
TARGET = 'Appliances'
DAILY_PERIOD = 24
TEST_DAYS = 14
MODEL_ID = 'amazon/chronos-bolt-small'  # try 'amazon/chronos-bolt-tiny' if this is slow on CPU
QUANTILE_LEVELS = [0.1, 0.5, 0.9]

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
for d in [DATA_DIR, OUTPUT_DIR / 'forecasts', OUTPUT_DIR / 'metrics', OUTPUT_DIR / 'figures']:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

## Load hourly data and split (self-contained)

In [ ]:
hourly_path = DATA_DIR / 'energydata_hourly.csv'

if hourly_path.exists():
    hourly = pd.read_csv(hourly_path, index_col=0, parse_dates=True)
else:
    raw = pd.read_csv(RAW_CSV_URL)
    raw['date'] = pd.to_datetime(raw['date'])
    raw = raw.set_index('date').sort_index()
    energy_cols = ['Appliances', 'lights']
    sensor_cols = [c for c in raw.columns if c not in energy_cols + ['rv1', 'rv2']]
    hourly = pd.concat([raw[energy_cols].resample('h').sum(),
                         raw[sensor_cols].resample('h').mean()], axis=1)
    hourly.to_csv(hourly_path)

def train_test_split_by_days(series, test_days=TEST_DAYS):
    test_steps = test_days * DAILY_PERIOD
    return series.iloc[:-test_steps], series.iloc[-test_steps:]

def evaluate_forecast(name, y_true, y_pred, y_train, seasonality=DAILY_PERIOD):
    y_pred = y_pred.reindex(y_true.index)
    valid = y_true.notna() & y_pred.notna()
    yt, yp = y_true.loc[valid], y_pred.loc[valid]
    y_train_f = y_train.astype(float)
    naive_err = np.abs(y_train_f.iloc[seasonality:].values - y_train_f.iloc[:-seasonality].values)
    scale = naive_err.mean()
    return {
        'model': name,
        'MAE': float(np.mean(np.abs(yt.values - yp.values))),
        'RMSE': float(np.sqrt(np.mean((yt.values - yp.values) ** 2))),
        'MASE': float(np.mean(np.abs(yt.values - yp.values)) / scale) if scale else float('nan'),
        'Bias': float(np.mean(yp.values - yt.values)),
        'n_points': int(valid.sum()),
    }

series = hourly[TARGET].asfreq('h')
train, test = train_test_split_by_days(series, TEST_DAYS)
print(f'Train: {train.index.min()} to {train.index.max()} ({len(train)} obs)')
print(f'Test:  {test.index.min()} to {test.index.max()} ({len(test)} obs)')

## Load the pretrained model

In [ ]:
pipeline = BaseChronosPipeline.from_pretrained(MODEL_ID, device_map='cpu')
print(f'Loaded {MODEL_ID}')

## Forecast helper
`predict_quantiles` takes the training series as-is (no gradient updates — pure in-context inference) and returns both quantile and mean forecasts.

In [ ]:
def chronos_forecast(pipeline, context, horizon, quantile_levels=QUANTILE_LEVELS):
    inputs = torch.tensor(context.values, dtype=torch.float32)
    quantiles, mean = pipeline.predict_quantiles(
        inputs=inputs, prediction_length=horizon, quantile_levels=quantile_levels,
    )
    mean = mean[0].numpy()
    quantile_frame = pd.DataFrame(
        quantiles[0].numpy(), columns=[f'q{int(q * 100)}' for q in quantile_levels]
    )
    return mean, quantile_frame

## Literal 24-hour-ahead forecast

In [ ]:
mean_24h, quantiles_24h = chronos_forecast(pipeline, train, DAILY_PERIOD)
actual_24h = test.iloc[:DAILY_PERIOD]
quantiles_24h.index = actual_24h.index
rmse_24h = float(np.sqrt(np.mean((actual_24h.values - mean_24h[:len(actual_24h)]) ** 2)))
print(f'24h-ahead RMSE: {rmse_24h:.1f} Wh')

# pandas' .plot() is used for every artist here (not raw ax.plot()) so they all
# share one datetime-axis converter — mixing the two silently drops a line.
mean_series_24h = pd.Series(mean_24h[:len(actual_24h)], index=actual_24h.index)
q10_24h = pd.Series(quantiles_24h['q10'].values[:len(actual_24h)], index=actual_24h.index)
q90_24h = pd.Series(quantiles_24h['q90'].values[:len(actual_24h)], index=actual_24h.index)

fig, ax = plt.subplots(figsize=(12, 4))
train.tail(3 * DAILY_PERIOD).plot(ax=ax, label='Train (last 3 days)', color='#999999')
actual_24h.plot(ax=ax, label='Actual (next 24h)', color='black', linewidth=1.8)
mean_series_24h.plot(ax=ax, label='Chronos-Bolt forecast', color='#8a1f6e', linewidth=1.6)
ax.fill_between(actual_24h.index, q10_24h.values, q90_24h.values,
                color='#8a1f6e', alpha=0.15, label='10-90% quantile band')
ax.set_title('Chronos-Bolt (zero-shot) — literal 24-hour-ahead forecast')
ax.set_xlabel('Date'); ax.set_ylabel('Appliances (Wh / hour)')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '07a_chronos_24h_forecast.png')
plt.show()

## Full 336-hour continuous test-period forecast
(for comparability with every other part)

In [ ]:
mean_full, quantiles_full = chronos_forecast(pipeline, train, len(test))
mean_full_series = pd.Series(mean_full, index=test.index, name='foundation_model')
quantiles_full.index = test.index

metrics = evaluate_forecast('chronos_bolt_zeroshot', test, mean_full_series, train)
print('Chronos-Bolt metrics over the full 336h test period:')
pd.Series(metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
train.tail(7 * DAILY_PERIOD).plot(ax=ax, label='Train (last 7 days)', color='#999999', linewidth=1)
test.plot(ax=ax, label='Actual (test)', color='black', linewidth=1.6)
mean_full_series.plot(ax=ax, label='Chronos-Bolt (336h, zero-shot)', color='#8a1f6e', linewidth=1.2)
ax.fill_between(test.index, quantiles_full['q10'].values, quantiles_full['q90'].values,
                color='#8a1f6e', alpha=0.12, label='10-90% quantile band')
ax.set_title('Chronos-Bolt vs. actual — full 336h continuous test-period forecast (zero-shot)')
ax.set_xlabel('Date'); ax.set_ylabel('Appliances (Wh / hour)')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '07b_chronos_full_test_forecast.png')
plt.show()

## Save outputs

In [ ]:
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / 'metrics' / 'chronos_metrics.csv', index=False)
pd.DataFrame({'actual': test, 'foundation_model': mean_full_series}).to_csv(
    OUTPUT_DIR / 'forecasts' / 'chronos_forecasts.csv'
)
quantiles_full.to_csv(OUTPUT_DIR / 'forecasts' / 'chronos_quantiles.csv')
print('Saved forecasts, quantiles, and metrics to outputs/')